# 03 · Leakage-Free 5-Fold Split

**Problem:** many patches were cut from the **same foot photo**, sometimes overlapping, and some files are copies.
If related patches land in both training and test, the test becomes too easy (**data leakage**).

**Goal:** put every related patch in **one group**, and keep each group entirely inside one fold.

**Methods**
1. **SIFT keypoints (OpenCV):** distinctive small points (wound edges, skin creases) with descriptors that survive scaling and rotation.
2. **Candidate search:** a FLANN nearest-neighbour index over all descriptors finds which photos or patches share many keypoints with each patch.
3. **Verification:** pairwise ratio test plus **RANSAC** checks the matches fit one consistent crop (a similarity transform). We accept a link only with at least `MIN_INLIERS` agreeing points **and** a plausible transform scale. Degenerate fits, where all points collapse to one spot, were found to be false links.
4. **Grouping:** union-find joins (a) patch-to-photo links, (b) patch-to-patch links (overlapping crops), (c) exact copies (MD5) and (d) near-duplicates from notebook 02.
5. **Split:** `StratifiedGroupKFold` (scikit-learn) gives 5 folds. Groups never cross folds. We try 300 random seeds and keep the one with the most balanced fold sizes and ulcer ratios. This uses only labels and group sizes, never model results.

**Output:** `splits.csv`, used by every training notebook.

In [ ]:
import hashlib, time
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

PATCHES = next(Path("/kaggle/input").rglob("Patches"))
DATA = PATCHES.parent
ORIGINALS = DATA / "Original Images"
CLASS_DIRS = ["Abnormal(Ulcer)", "Normal(Healthy skin)"]
EXTS = {".jpg", ".jpeg", ".png", ".gif", ".webp"}
MIN_INLIERS = 10      # minimum RANSAC-consistent keypoints to accept a link
MAX_SIDE = 1400       # originals are resized so the long side is at most this
SEED = 42

def images(folder):
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in EXTS)

def read_gray(p):
    a = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    return a if a is not None else np.asarray(Image.open(p).convert("L"))

dup_csv = next(Path("/kaggle/input").rglob("kdfu_duplicate_table.csv"), None)
print("Dataset:", DATA)
print("Notebook 02 table:", dup_csv)
print("CPU cores:", __import__("os").cpu_count())

## 1. SIFT features for original photos and patches
Patches are upscaled 2x and use a lower contrast threshold, so smooth skin still yields keypoints.

In [ ]:
osift = cv2.SIFT_create(nfeatures=2500)
psift = cv2.SIFT_create(contrastThreshold=0.02)

def features(gray, sift):
    kp, d = sift.detectAndCompute(gray, None)
    if d is None:
        return np.zeros((0, 2), np.float32), None
    return np.float32([k.pt for k in kp]), d

nodes, kps, descs = [], [], []
t = time.time()
for p in images(ORIGINALS):
    g = read_gray(p)
    s = min(1.0, MAX_SIDE / max(g.shape))
    if s < 1:
        g = cv2.resize(g, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    k, d = features(g, osift)
    nodes.append({"kind": "photo", "cls": "", "file": p.name, "path": p, "scale": s})
    kps.append(k); descs.append(d)
N_PHOTO = len(nodes)

for cls in CLASS_DIRS:
    for p in images(PATCHES / cls):
        g = cv2.resize(read_gray(p), None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        k, d = features(g, psift)
        nodes.append({"kind": "patch", "cls": cls, "file": p.name, "path": p,
                      "md5": hashlib.md5(p.read_bytes()).hexdigest()})
        kps.append(k); descs.append(d)
PATCH_IDS = list(range(N_PHOTO, len(nodes)))

n_kp = pd.Series([len(kps[i]) for i in PATCH_IDS])
print(f"{N_PHOTO} photos, {len(PATCH_IDS)} patches, {time.time() - t:.0f}s")
print("Keypoints per patch: median", int(n_kp.median()), "| patches with < 10 keypoints:", int((n_kp < 10).sum()))

## 2. Candidate search (FLANN index over all descriptors)
For each patch, each descriptor votes for the images owning its 6 nearest neighbours. Copies of the patch itself are excluded, and the top 8 images become candidates.

In [ ]:
valid = [i for i, d in enumerate(descs) if d is not None]
owner = np.concatenate([np.full(len(descs[i]), i) for i in valid])
flann = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=64))
flann.add([np.vstack([descs[i] for i in valid]).astype(np.float32)])
flann.train()

md5_of = {i: nodes[i]["md5"] for i in PATCH_IDS}
t = time.time()
candidates = {}
for i in PATCH_IDS:
    if descs[i] is None or len(descs[i]) < 5:
        candidates[i] = []
        continue
    votes = Counter()
    for nn in flann.knnMatch(descs[i].astype(np.float32), k=6):
        for o in {owner[m.trainIdx] for m in nn}:
            if o != i and md5_of.get(o) != md5_of[i]:
                votes[o] += 1
    candidates[i] = [c for c, _ in votes.most_common(8)]
print(f"Candidate search: {time.time() - t:.0f}s")

## 3. Verification (ratio test + RANSAC), run in parallel on all CPU cores

In [ ]:
def verify(i, d_i, k_i, cands):
    cv2.setRNGSeed(SEED + i)        # makes RANSAC repeatable run to run
    bf = cv2.BFMatcher()
    out = []
    for c, d_c, k_c in cands:
        if d_c is None or len(d_c) < 5:
            continue
        good = [m for m, n in (y for y in bf.knnMatch(d_i, d_c, k=2) if len(y) == 2) if m.distance < 0.8 * n.distance]
        if len(good) < MIN_INLIERS:
            continue
        src = np.float32([k_i[m.queryIdx] for m in good])
        dst = np.float32([k_c[m.trainIdx] for m in good])
        M, inl = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC, ransacReprojThreshold=6.0)
        n_in = int(inl.sum()) if inl is not None else 0
        if n_in >= MIN_INLIERS:
            out.append((i, c, n_in, M.ravel().tolist()))
    return out

t = time.time()
results = Parallel(n_jobs=-1, batch_size=4)(
    delayed(verify)(i, descs[i], kps[i], [(c, descs[c], kps[c]) for c in candidates[i]])
    for i in PATCH_IDS if candidates[i])
edges = pd.DataFrame([e for r in results for e in r], columns=["a", "b", "inliers", "M"])
edges["type"] = np.where(edges.b < N_PHOTO, "patch-photo", "patch-patch")
print(f"Verification: {time.time() - t:.0f}s, links found: {len(edges)}")
edges.to_pickle("sift_links_raw.pkl")   # saved early so a later error does not lose 20 minutes of work

# Reject geometrically implausible links. A visual check of v2 showed that every false link
# had a degenerate transform (scale ~ 0: RANSAC collapsed all points onto one spot).
# Patch->patch crops have similar size (scale ~1); patch->photo scale depends on photo resizing.
M_all = np.stack(edges.M.apply(np.array)).reshape(-1, 2, 3)
edges["scale"] = np.sqrt(np.abs(np.linalg.det(M_all[:, :, :2])))
plausible = np.where(edges.type == "patch-photo", edges.scale >= 0.1, edges.scale.between(0.4, 2.5))
rejected = edges[~plausible].reset_index(drop=True)
edges = edges[plausible].reset_index(drop=True)
print(f"Rejected {len(rejected)} links with implausible scale; kept {len(edges)}")
edges.groupby("type").inliers.describe()

## 4. Visual check of the links
The green box shows where the patch sits in the photo. **Check the borderline rows carefully.** The last plot shows links that were *rejected*, which should look like different wounds.

In [ ]:
def photo_with_box(edge):
    ph = nodes[edge.b]
    im = np.array(Image.open(ph["path"]).convert("RGB"))      # np.array makes a writable copy
    if ph["scale"] < 1:
        im = cv2.resize(im, None, fx=ph["scale"], fy=ph["scale"], interpolation=cv2.INTER_AREA)
    M = np.array(edge.M).reshape(2, 3)
    corners = np.float32([[0, 0], [448, 0], [448, 448], [0, 448]])   # patch was upscaled to 448
    pts = (M[:, :2] @ corners.T).T + M[:, 2]
    cv2.polylines(im, [pts.astype(np.int32)], True, (0, 255, 0), max(3, im.shape[1] // 150))
    return im

def show_links(df, title, k=6, seed=0):
    df = df.sample(min(k, len(df)), random_state=seed)
    fig, axes = plt.subplots(len(df), 2, figsize=(8, 3.3 * len(df)), squeeze=False)
    for r, e in enumerate(df.itertuples()):
        a = nodes[e.a]
        axes[r, 0].imshow(Image.open(a["path"]).convert("RGB"))
        axes[r, 0].set_title(f"{a['cls'][:6]}/{a['file']}", fontsize=9)
        if e.type == "patch-photo":
            axes[r, 1].imshow(photo_with_box(e))
        else:
            axes[r, 1].imshow(Image.open(nodes[e.b]["path"]).convert("RGB"))
        axes[r, 1].set_title(f"{nodes[e.b]['file'][:40]}  inliers={e.inliers}", fontsize=9)
        for ax in axes[r]:
            ax.axis("off")
    fig.suptitle(title, fontweight="bold"); plt.tight_layout(); plt.show()

pp, ppa = edges[edges.type == "patch-photo"], edges[edges.type == "patch-patch"]
show_links(pp[pp.inliers >= 30], "Patch -> photo, strong (>= 30 inliers)")
show_links(pp[pp.inliers < 20], f"Patch -> photo, borderline ({MIN_INLIERS}-19 inliers)")
show_links(ppa[ppa.inliers < 20], f"Patch -> patch, borderline ({MIN_INLIERS}-19 inliers)")
if len(rejected):
    show_links(rejected[rejected.type == "patch-patch"], "REJECTED links (implausible scale): these should look WRONG")

## 5. Build groups
Links, exact copies and notebook 02's near-duplicate groups are all joined with union-find.

In [ ]:
parent = list(range(len(nodes)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    parent[find(a)] = find(b)

for e in edges.itertuples():                       # (a) + (b) SIFT links
    union(e.a, e.b)
first = {}
for i in PATCH_IDS:                                # (c) exact copies
    h = nodes[i]["md5"]
    if h in first: union(i, first[h])
    else: first[h] = i
if dup_csv is not None:                            # (d) near-duplicates from notebook 02
    dup = pd.read_csv(dup_csv)
    key = {(nodes[i]["cls"], nodes[i]["file"]): i for i in PATCH_IDS}
    for _, g in dup[dup.dup_group > 0].groupby("dup_group"):
        ids = [key[(c, f)] for c, f in zip(g["class"], g["file"])]
        for j in ids[1:]:
            union(ids[0], j)

linked_photo = set(edges.loc[edges.type == "patch-photo", "a"])
df = pd.DataFrame([{
    "class": nodes[i]["cls"], "file": nodes[i]["file"], "label": int(nodes[i]["cls"].startswith("Abnormal")),
    "md5": nodes[i]["md5"], "group": find(i), "matched_photo": i in linked_photo,
} for i in PATCH_IDS])
sizes = df.group.value_counts()
print("Patches:", len(df), "| groups:", df.group.nunique(), "| single-patch groups:", int((sizes == 1).sum()),
      "| largest group:", int(sizes.max()))
print("Patches linked to an original photo:", df.groupby("class").matched_photo.sum().to_dict())
print("Groups containing both ulcer and normal patches:", int((df.groupby("group").label.nunique() > 1).sum()))

# Look inside the largest group: it should be one patient / one wound, not unrelated images
big = df[(df.group == sizes.index[0]) & ~df.md5.duplicated()].head(24)
fig, axes = plt.subplots(3, 8, figsize=(16, 6.3))
for ax, r in zip(axes.ravel(), list(big.itertuples()) + [None] * 24):
    ax.axis("off")
    if r is not None:
        ax.imshow(Image.open(PATCHES / r._1 / r.file).convert("RGB")); ax.set_title(r.file, fontsize=8)
fig.suptitle(f"Largest group: {int(sizes.iloc[0])} files (first 24 unique shown)", fontweight="bold")
plt.tight_layout(); plt.show()

## 6. Stratified group 5-fold split

In [ ]:
df["keep"] = ~df.duplicated("md5")    # use one file per exact copy for training/testing

def make_folds(seed):
    folds = np.full(len(df), -1)
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for k, (_, test_idx) in enumerate(sgkf.split(df, df.label, df.group)):
        folds[test_idx] = k
    return folds

def imbalance(folds):
    u = df.assign(fold=folds)[df.keep]
    n, ulcer_pct = u.groupby("fold").size(), 100 * u.groupby("fold").label.mean()
    return n.std() / n.mean() * 100 + ulcer_pct.std()   # % spread of fold sizes + spread of ulcer %

scores = {seed: imbalance(make_folds(seed)) for seed in range(300)}
best_seed = min(scores, key=scores.get)
df["fold"] = make_folds(best_seed)
print(f"Most balanced seed: {best_seed} (imbalance score {scores[best_seed]:.2f}; median over seeds {np.median(list(scores.values())):.2f})")

# Safety checks: nothing related may cross folds
assert (df.groupby("group").fold.nunique() == 1).all(), "a group crosses folds"
assert (df.groupby("md5").fold.nunique() == 1).all(), "an exact copy crosses folds"

u = df[df.keep]
fold_stats = pd.DataFrame({
    "files": df.groupby("fold").size(),
    "unique images": u.groupby("fold").size(),
    "ulcer (unique)": u[u.label == 1].groupby("fold").size(),
    "normal (unique)": u[u.label == 0].groupby("fold").size(),
    "groups": df.groupby("fold").group.nunique(),
})
fold_stats["ulcer %"] = (100 * fold_stats["ulcer (unique)"] / fold_stats["unique images"]).round(1)
print("All safety checks passed: no group or exact copy appears in more than one fold.")
fold_stats

In [ ]:
df.to_csv("splits.csv", index=False)
edges.drop(columns="M").assign(a_file=lambda e: [nodes[i]["file"] for i in e.a],
                               b_file=lambda e: [nodes[i]["file"] for i in e.b]).to_csv("sift_links.csv", index=False)
print("Saved splits.csv and sift_links.csv")
print(f"Unique images used for training/testing: {int(df.keep.sum())} "
      f"(ulcer {int(df[df.keep].label.sum())}, normal {int((df[df.keep].label == 0).sum())})")
df.head()

## How later notebooks use `splits.csv`
- For each run *k* = 0..4: **test = fold k**, **validation = fold (k+1) mod 5** (for early stopping), **training = the other 3 folds**.
- Only rows with `keep == True` are used (one file per exact copy).
- Every model uses the same file, so comparisons are fair.

**Summary:** the numbers printed above (groups, photo links, unique images per fold) are this notebook's results. Check the borderline visual checks before relying on the split.